# 06 — Supervised Learning: Regression

**Topics:** LinearRegression, Ridge, Lasso, ElasticNet, cross-validation, regression metrics, regularization path.

**Reference:** [sklearn linear models](https://scikit-learn.org/stable/modules/linear_model.html)

**Dataset:** California Housing — predicting median house values.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

housing = fetch_california_housing(as_frame=True)
X = housing.data.copy()
y = housing.target.copy()  # Median house value in $100k

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X.shape)
X.head()

---
## Exercise 1 — Baseline: Linear Regression

1. Build a pipeline: `StandardScaler` → `LinearRegression`.
2. Fit on train, predict on test.
3. Compute and store: `rmse`, `mae`, `r2` (on test set).
4. Implement `regression_report(y_true, y_pred)` that prints and returns a dict with these 3 metrics, rounded to 4dp.

In [ ]:
def regression_report(y_true, y_pred) -> dict:
    """
    Returns dict: rmse, mae, r2 — all rounded to 4dp.
    """
    # YOUR CODE HERE
    pass

# YOUR CODE HERE: build pipeline, fit, predict, report
lr_pipeline = None
lr_metrics = None

In [ ]:
# --- ASSERTIONS ---
assert set(lr_metrics.keys()) == {'rmse', 'mae', 'r2'}
assert lr_metrics['r2'] > 0.55, "R2 must be above 0.55 for a reasonable baseline"
assert lr_metrics['rmse'] > 0
assert lr_metrics['mae'] < lr_metrics['rmse'], "MAE should be less than RMSE"
print(f"✓ Exercise 1 passed")
print(lr_metrics)

---
## Exercise 2 — Ridge Regression & Alpha Tuning

1. Train Ridge regression for `alpha` values: `[0.01, 0.1, 1, 10, 100, 1000]`.
2. For each alpha, compute **5-fold CV RMSE** (negative MSE scoring → convert to RMSE).
3. Return `ridge_results`: DataFrame with columns `alpha`, `cv_rmse_mean`, `cv_rmse_std`.
4. Identify `best_alpha_ridge`: the alpha with lowest `cv_rmse_mean`.
5. Train final Ridge model with `best_alpha_ridge`, report test metrics.

In [ ]:
def tune_ridge(X_train, y_train, alphas):
    """
    Returns (ridge_results DataFrame, best_alpha_ridge float).
    """
    # YOUR CODE HERE
    pass

alphas = [0.01, 0.1, 1, 10, 100, 1000]
ridge_results, best_alpha_ridge = tune_ridge(X_train, y_train, alphas)

In [ ]:
# --- ASSERTIONS ---
assert list(ridge_results.columns) == ['alpha', 'cv_rmse_mean', 'cv_rmse_std']
assert len(ridge_results) == len(alphas)
assert best_alpha_ridge in alphas
assert (ridge_results['cv_rmse_mean'] > 0).all()
print(f"✓ Exercise 2 passed — Best alpha: {best_alpha_ridge}")
print(ridge_results)

---
## Exercise 3 — Lasso & Feature Selection via Regularization

Lasso performs implicit feature selection by driving some coefficients to exactly zero.

1. Use `LassoCV` with 5-fold CV to find the best alpha automatically.
2. Fit on training data (with StandardScaler in pipeline).
3. Extract coefficients and identify:
   - `nonzero_features`: list of feature names with non-zero coefficients.
   - `zeroed_features`: list of features Lasso zeroed out.
4. Build `lasso_coeff_df`: DataFrame with `feature`, `coefficient`, sorted by `abs(coefficient)` descending.

In [ ]:
def fit_lasso_cv(X_train, y_train, feature_names):
    """
    Returns (lasso_pipeline, best_alpha, nonzero_features, zeroed_features, lasso_coeff_df)
    """
    # YOUR CODE HERE
    pass

lasso_pipeline, best_alpha_lasso, nonzero_features, zeroed_features, lasso_coeff_df = fit_lasso_cv(
    X_train, y_train, X.columns.tolist()
)

In [ ]:
# --- ASSERTIONS ---
assert isinstance(nonzero_features, list)
assert isinstance(zeroed_features, list)
assert set(nonzero_features + zeroed_features) == set(X.columns), "All features must be accounted for"
assert list(lasso_coeff_df.columns) == ['feature', 'coefficient']
assert lasso_coeff_df['coefficient'].abs().is_monotonic_decreasing
print(f"✓ Exercise 3 passed — best alpha: {best_alpha_lasso:.6f}")
print(f"Non-zero features: {nonzero_features}")
print(f"Zeroed features: {zeroed_features}")

---
## Exercise 4 — ElasticNet & Regularization Path

1. Train `ElasticNet` with `l1_ratio` values `[0.1, 0.5, 0.7, 0.9, 0.95, 1.0]` and `alpha=0.1`.
2. For each `l1_ratio`, record: `n_nonzero_coefs`, `train_r2`, `test_r2`.
3. Return `elasticnet_path`: DataFrame with columns `l1_ratio`, `n_nonzero_coefs`, `train_r2`, `test_r2`.
4. Explain in a markdown cell: what does increasing `l1_ratio` toward 1.0 do to the model?

In [ ]:
def elasticnet_path_analysis(X_train, y_train, X_test, y_test, l1_ratios):
    """
    Returns elasticnet_path DataFrame.
    """
    # YOUR CODE HERE
    pass

l1_ratios = [0.1, 0.5, 0.7, 0.9, 0.95, 1.0]
elasticnet_path = elasticnet_path_analysis(X_train, y_train, X_test, y_test, l1_ratios)

In [ ]:
# --- ASSERTIONS ---
assert list(elasticnet_path.columns) == ['l1_ratio', 'n_nonzero_coefs', 'train_r2', 'test_r2']
assert len(elasticnet_path) == len(l1_ratios)
print("✓ Exercise 4 passed")
print(elasticnet_path)

**Your answer:** As `l1_ratio` approaches 1.0, ElasticNet becomes more like...

*(Write your explanation here)*

---
## Exercise 5 — Cross-Validation Deep Dive

**Task:** Understand what CV scores actually tell you.

1. Run 5-fold and 10-fold CV for `LinearRegression` on the full dataset (scaled).
2. Compute for each fold count: `fold_r2_scores` (array of fold scores), `mean_r2`, `std_r2`.
3. Compare: is the model's score consistent across folds, or is there high variance?
4. Build `cv_comparison_df`: columns `n_folds`, `mean_r2`, `std_r2`, `min_fold`, `max_fold`.
5. Also compute the **learning curve** manually: train sizes `[0.1, 0.2, 0.4, 0.6, 0.8, 1.0]`. For each size, train on that fraction and record train R2 and validation R2 (using the rest as validation). Return as `learning_curve_df`.

In [ ]:
def run_cv_analysis(X, y):
    """
    Returns (cv_comparison_df, learning_curve_df)
    """
    # YOUR CODE HERE
    pass

cv_comparison_df, learning_curve_df = run_cv_analysis(X, y)

In [ ]:
# --- ASSERTIONS ---
assert list(cv_comparison_df.columns) == ['n_folds', 'mean_r2', 'std_r2', 'min_fold', 'max_fold']
assert set(cv_comparison_df['n_folds']) == {5, 10}
assert list(learning_curve_df.columns) == ['train_size', 'train_r2', 'val_r2']
assert len(learning_curve_df) == 6
print("✓ Exercise 5 passed")
print(cv_comparison_df)
print(learning_curve_df)

---
## Exercise 6 — Residual Analysis

**Task:** Diagnose model failures through residual analysis — essential for any regression interview.

Using your best model so far:

1. Compute `residuals = y_test - y_pred`.
2. Return `residual_stats`: dict with `mean`, `std`, `skewness` (use the formula, not scipy), `max_abs_error`.
3. Identify `large_error_mask`: boolean Series, `True` for test samples where `|residual| > 2 * std(residuals)`.
4. Build `error_analysis_df`: the test features for those large-error samples, plus `y_true` and `residual` columns.
5. What feature values are common in the high-error samples? Write your observation as a markdown comment.

In [ ]:
def analyze_residuals(model, X_test, y_test):
    """
    Returns (residual_stats dict, error_analysis_df DataFrame)
    """
    # YOUR CODE HERE
    pass

residual_stats, error_analysis_df = analyze_residuals(lr_pipeline, X_test, y_test)

In [ ]:
# --- ASSERTIONS ---
assert set(residual_stats.keys()) == {'mean', 'std', 'skewness', 'max_abs_error'}
assert abs(residual_stats['mean']) < 1, "Residual mean should be near 0 for a decent model"
assert 'y_true' in error_analysis_df.columns
assert 'residual' in error_analysis_df.columns
print(f"✓ Exercise 6 passed — {len(error_analysis_df)} large-error samples")
print(residual_stats)

---
## Exercise 7 — Model Comparison Table

**Task:** Produce a clean comparison of all 4 model families.

Build `model_comparison`: DataFrame comparing LinearRegression, Ridge (best alpha), Lasso (CV), and ElasticNet (best l1_ratio, alpha=0.1) on:
- `test_rmse`, `test_mae`, `test_r2`
- `n_nonzero_coefs` (LinearRegression and Ridge = all features)
- `cv_r2_mean` (5-fold CV on training set)

Sort by `test_r2` descending. Index = model name.

In [ ]:
def build_model_comparison(X_train, y_train, X_test, y_test):
    """
    Returns model_comparison DataFrame.
    """
    # YOUR CODE HERE
    pass

model_comparison = build_model_comparison(X_train, y_train, X_test, y_test)

In [ ]:
# --- ASSERTIONS ---
assert set(model_comparison.index) == {'LinearRegression', 'Ridge', 'Lasso', 'ElasticNet'}
for col in ['test_rmse', 'test_mae', 'test_r2', 'n_nonzero_coefs', 'cv_r2_mean']:
    assert col in model_comparison.columns, f"Missing column: {col}"
assert model_comparison['test_r2'].is_monotonic_decreasing
print("✓ Exercise 7 passed")
print(model_comparison)